# fMRI Decomposition Visualization

Visualize PCA and ICA/ICASSO decomposition results:
- Spatial maps
- Component timeseries  
- Stability scores (for ICASSO)
- Variance explained
- Comparison across methods

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import seaborn as sns

# Setup plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

## Configuration

Set paths to your decomposition results:

In [ ]:
# Path to results directory
RESULTS_DIR = Path('../test_data/small_validation_afni_data')

# Prefix for results (e.g., 'vis_icasso' loads vis_icasso_icasso_maps.nii.gz, etc.)
RESULT_PREFIX = 'vis_icasso'

# Method: 'pca', 'ica', or 'icasso'
METHOD = 'icasso'

# Original data for reconstruction testing
FMRI_FILE = RESULTS_DIR / 'vis_small_test_r01.nii.gz'
MASK_FILE = RESULTS_DIR / 'vis_mask.nii.gz'

## Load Data

In [ ]:
# Load original fMRI data
fmri_img = nib.load(FMRI_FILE)
fmri_data = fmri_img.get_fdata()
n_timepoints = fmri_data.shape[-1]

# Load mask
mask_img = nib.load(MASK_FILE)
mask_data = mask_img.get_fdata()
mask_bool = mask_data > 0
n_voxels = mask_bool.sum()

# Reshape fMRI to 2D (timepoints x voxels)
fmri_2d = np.zeros((n_timepoints, n_voxels))
for t in range(n_timepoints):
    fmri_2d[t] = fmri_data[..., t][mask_bool]

print(f"fMRI data: {fmri_2d.shape} (timepoints x voxels)")
print(f"Mask: {mask_bool.sum()} voxels")

In [ ]:
# Load decomposition results
maps_file = RESULTS_DIR / f"{RESULT_PREFIX}_{METHOD}_maps.nii.gz"
timeseries_file = RESULTS_DIR / f"{RESULT_PREFIX}_{METHOD}_timeseries.1D"

# Load spatial maps (4D)
maps_img = nib.load(maps_file)
maps_data = maps_img.get_fdata()
n_components = maps_data.shape[-1]

# Reshape maps to 2D (components x voxels)
maps_2d = np.zeros((n_components, n_voxels))
for i in range(n_components):
    maps_2d[i] = maps_data[..., i][mask_bool]

# Load timeseries
timeseries = np.loadtxt(timeseries_file)
if timeseries.ndim == 1:
    timeseries = timeseries.reshape(-1, 1)

print(f"\nDecomposition ({METHOD.upper()}):")
print(f"  Components: {n_components}")
print(f"  Spatial maps: {maps_2d.shape}")
print(f"  Timeseries: {timeseries.shape}")

# Load additional files if they exist
stability_file = RESULTS_DIR / f"{RESULT_PREFIX}_{METHOD}_stability.1D"
variance_file = RESULTS_DIR / f"{RESULT_PREFIX}_{METHOD}_variance.1D"

stability = None
variance = None

if stability_file.exists():
    stability = np.loadtxt(stability_file)
    print(f"  Stability scores: {stability.shape}")
    print(f"    Mean: {stability.mean():.3f}, Range: [{stability.min():.3f}, {stability.max():.3f}]")

if variance_file.exists():
    variance = np.loadtxt(variance_file)
    print(f"  Variance explained: {variance.shape}")
    print(f"    Total: {variance.sum():.4f} ({variance.sum()*100:.2f}%)")
    print(f"    Per component: {variance.mean():.6f} ± {variance.std():.6f}")

## Investigate Reconstruction

In [ ]:
# Test reconstruction: original ≈ timeseries @ components
fmri_centered = fmri_2d - fmri_2d.mean(axis=0)
reconstructed = timeseries @ maps_2d

# Compute reconstruction error
mse = ((fmri_centered - reconstructed) ** 2).mean()
corr = np.corrcoef(fmri_centered.flatten(), reconstructed.flatten())[0, 1]

print(f"\nReconstruction Quality:")
print(f"  MSE: {mse:.6f}")
print(f"  Correlation: {corr:.4f}")

# Check component norms
map_norms = np.linalg.norm(maps_2d, axis=1)
ts_norms = np.linalg.norm(timeseries, axis=0)

print(f"\nComponent Norms:")
print(f"  Spatial maps: {map_norms.mean():.2f} ± {map_norms.std():.2f}")
print(f"  Timeseries: {ts_norms.mean():.2f} ± {ts_norms.std():.2f}")

# Compute variance explained per component (proper method)
total_var = (fmri_centered ** 2).sum()
var_per_comp = np.zeros(n_components)

for i in range(n_components):
    comp_reconstruction = timeseries[:, i:i+1] @ maps_2d[i:i+1, :]
    var_per_comp[i] = (comp_reconstruction ** 2).sum() / total_var

print(f"\nVariance Explained (recomputed):")
print(f"  Total: {var_per_comp.sum():.4f} ({var_per_comp.sum()*100:.2f}%)")
print(f"  Per component: {var_per_comp.mean():.4f} ± {var_per_comp.std():.4f}")
print(f"  Top 5: {var_per_comp[:5]}")

## Plot Spatial Maps

In [ ]:
def plot_spatial_map(component_idx, ax=None):
    """Plot spatial map for a component"""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
    # Get component map
    comp_map = maps_data[..., component_idx]
    
    # Find middle slice
    mid_slice = comp_map.shape[2] // 2
    slice_data = comp_map[:, :, mid_slice]
    
    # Plot
    vmax = np.abs(slice_data).max()
    im = ax.imshow(slice_data.T, cmap='seismic', vmin=-vmax, vmax=vmax,
                   origin='lower', aspect='equal')
    
    title = f"Component {component_idx}"
    if stability is not None:
        title += f" (stability: {stability[component_idx]:.3f})"
    if variance is not None:
        title += f"\nVar: {var_per_comp[component_idx]:.4f}"
    
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

# Plot first 6 components
n_show = min(6, n_components)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i in range(n_show):
    plot_spatial_map(i, ax=axes[i])

plt.tight_layout()
plt.show()

## Plot Timeseries

In [ ]:
# Plot timeseries for first 6 components
fig, axes = plt.subplots(n_show, 1, figsize=(12, 2*n_show))
if n_show == 1:
    axes = [axes]

for i in range(n_show):
    ax = axes[i]
    ax.plot(timeseries[:, i], linewidth=0.5)
    
    title = f"Component {i} Timeseries"
    if stability is not None:
        title += f" (stability: {stability[i]:.3f})"
    
    ax.set_title(title)
    ax.set_xlabel('Time (TRs)')
    ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Component Statistics

In [ ]:
# Plot variance explained
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variance per component
ax = axes[0]
ax.bar(range(n_components), var_per_comp)
ax.set_xlabel('Component')
ax.set_ylabel('Variance Explained')
ax.set_title(f'Variance Explained per Component\nTotal: {var_per_comp.sum():.2%}')
ax.grid(True, alpha=0.3)

# Cumulative variance
ax = axes[1]
cumsum_var = np.cumsum(var_per_comp)
ax.plot(range(n_components), cumsum_var, marker='o')
ax.axhline(y=0.85, color='r', linestyle='--', label='85%')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Variance Explained')
ax.set_title('Cumulative Variance Explained')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot stability scores (if available)
if stability is not None:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    
    ax.bar(range(n_components), stability)
    ax.axhline(y=0.7, color='r', linestyle='--', label='Min threshold (0.7)')
    ax.set_xlabel('Component')
    ax.set_ylabel('Stability Score')
    ax.set_title(f'ICASSO Stability Scores\nMean: {stability.mean():.3f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Summary Statistics

In [ ]:
print(f"\n{'='*60}")
print(f"DECOMPOSITION SUMMARY: {METHOD.upper()}")
print(f"{'='*60}")
print(f"\nData:")
print(f"  Shape: {fmri_2d.shape} (timepoints x voxels)")
print(f"  Mean: {fmri_2d.mean():.2f}, Std: {fmri_2d.std():.2f}")

print(f"\nComponents: {n_components}")
print(f"  Spatial maps range: [{maps_2d.min():.3f}, {maps_2d.max():.3f}]")
print(f"  Timeseries range: [{timeseries.min():.3f}, {timeseries.max():.3f}]")

print(f"\nReconstruction:")
print(f"  MSE: {mse:.6f}")
print(f"  Correlation: {corr:.4f}")
print(f"  Total variance explained: {var_per_comp.sum():.2%}")

if stability is not None:
    print(f"\nStability:")
    print(f"  Mean: {stability.mean():.3f}")
    print(f"  Components > 0.7: {(stability > 0.7).sum()}/{n_components}")
    print(f"  Components > 0.8: {(stability > 0.8).sum()}/{n_components}")
    print(f"  Components > 0.9: {(stability > 0.9).sum()}/{n_components}")

print(f"\n{'='*60}")